## Structured Output

Models can be requested to provide their response in a format matching a given schema types and methods for enforcing structured output.

## Pydantic

Pydantic models provide the richest feature set with field valadation. description, and nested structures.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")
from langchain_groq import ChatGroq
model= ChatGroq(model= "qwen/qwen3.6-27b")


In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie on a scale of 1 to 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x0000024F878EB380>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000024F88A581A0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The rating of the movie on a scale of 1 to 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': '

In [5]:
model.invoke("Provide The deails about the movie Inception")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request:** The user is asking for "details about the movie Inception". This is a straightforward request for information about a well-known film.\n\n2.  **Identify Key Information Needed:** For a movie, typical details include:\n   - Title\n   - Release year\n   - Director\n   - Writer(s)\n   - Screenplay/Story credits\n   - Genre\n   - Runtime\n   - Main cast\n   - Plot summary\n   - Themes/Concepts\n   - Box office/Reception\n   - Awards/Nominations\n   - Notable facts/Trivia\n   - Cultural impact\n\n3.  **Gather Accurate Information (Internal Knowledge):**\n   - *Title:* Inception\n   - *Release Year:* 2010\n   - *Director:* Christopher Nolan\n   - *Writer:* Christopher Nolan\n   - *Genre:* Sci-fi, Action, Thriller, Heist\n   - *Runtime:* ~148 minutes (2h 28m)\n   - *Main Cast:* Leonardo DiCaprio (Dom Cobb), Joseph Gordon-Levitt (Arthur), Elliot Page (Ariadne), Tom Hardy (Eames), Ken Watanabe (Saito),

In [7]:
model_with_structure.invoke("Provide The deails about the movie Inception")


Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message Output alongside Parsed structure

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(...,description="The year the movie was released")
    director: str = Field(...,description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie on a scale of 1 to 10") 
    
model_with_structure= model.with_structured_output(Movie, include_raw=True)

response= model_with_structure.invoke("Provide details about the movie Inception")

response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "Provide details about the movie Inception"\n   - I need to provide details about the movie "Inception".\n\n2.  **Identify Required Information:**\n   - The available tool is `Movie` which requires: `title`, `year`, `director`, `rating` (all required).\n   - I know the movie "Inception":\n     - Title: Inception\n     - Year: 2010\n     - Director: Christopher Nolan\n     - Rating: I need to provide a rating on a scale of 1 to 10. I\'ll use a generally accepted rating like 8.8 (IMDb) or just pick a reasonable value like 8.8 or 9.0. Since it\'s a required parameter, I\'ll use 8.8.\n\n3.  **Check Tool Parameters:**\n   - `title`: "Inception"\n   - `year`: 2010\n   - `director`: "Christopher Nolan"\n   - `rating`: 8.8 (or similar)\n   - All required parameters are available.\n\n4.  **Formulate Tool Call:**\n   - Call the `Movie` function with t

## Nested Structure

In [7]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str 
    year: int 
    director: str 
    rating: float 
    actors: list[Actor] 
    budget: float | None = Field(default=None, description="The budget of the movie in millions of dollars")

In [15]:
model_with_structure= model.with_structured_output(MovieDetails, include_raw=True)

response= model_with_structure.invoke("Provide details about the movie Inception")

print(response)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants details about the movie "Inception".\nI need to use the `MovieDetails` function to retrieve this information.\nThe required parameters for `MovieDetails` are:\n- title (string)\n- year (integer)\n- director (string)\n- rating (number)\n- actors (array of objects with name and role)\n\nSince the user didn\'t provide the specific details, I have to rely on my internal knowledge about the movie "Inception" to fill in the parameters.\nInception was released in 2010.\nDirector: Christopher Nolan.\nRating: 8.8 (IMDb)\nActors: Leonardo DiCaprio (Cobb), Joseph Gordon-Levitt (Arthur), Ellen Page (Ariadne), Tom Hardy (Eames), Ken Watanabe (Saito), Marion Cotillard (Mal), Cillian Murphy (Robert Fischer), Tom Berenger (Browning), Michael Caine (Miles).\nBudget: ~160 million USD (optional parameter)\n\nI will construct the function call with these values.\nTitle: "Inception"\nYear: 2010\nDirector: "Christopher Nol

## TypeDict

TypDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [17]:
from typing_extensions import TypedDict, Annotated

In [23]:
class MovieDict(TypedDict):
    """ A movie with details."""
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [25]:
model_with_typedict= model.with_structured_output(MovieDict)

response= model_with_typedict.invoke("Provide details about the movie avengers")

print(response)

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}


## DataClasses

A data class is a class typically containing mainly data, although there are't really any restrictions. You create it using the @dataclass decorator

In [30]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str


agent= create_agent(
    model, 
    response_format=ToolStrategy(ContactInfo)  # Auto-selects ProviderStrategy
    )

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from : john Doe, john.doe@example.com, 555-1234"
    }]
})

result

{'messages': [HumanMessage(content='Extract contact info from : john Doe, john.doe@example.com, 555-1234', additional_kwargs={}, response_metadata={}, id='3fb397f4-e854-4759-8380-9c4e4833577a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User wants to extract contact info from the string: "john Doe, john.doe@example.com, 555-1234"\n   - The input contains a name, an email, and a phone number, separated by commas.\n\n2.  **Identify Required Parameters for the Tool:**\n   - Tool: `ContactInfo`\n   - Parameters: `name` (string), `email` (string), `phone` (string)\n   - All are required.\n\n3.  **Map Input to Parameters:**\n   - `name`: "john Doe"\n   - `email`: "john.doe@example.com"\n   - `phone`: "555-1234"\n\n4.  **Construct Tool Call:**\n   - Call `ContactInfo` with the extracted values.\n\n5.  **Execute Tool Call:**\n   - `ContactInfo(name="john Doe", email="john.doe@example.com", phone="555-1234")

In [31]:
## Typedict
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from typing_extensions import TypedDict, Annotated

class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str


agent= create_agent(
    model, 
    response_format=ToolStrategy(ContactInfo)  # Auto-selects ProviderStrategy
    )

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from : john Doe, john.doe@example.com, 555-1234"
    }]
})

result

{'messages': [HumanMessage(content='Extract contact info from : john Doe, john.doe@example.com, 555-1234', additional_kwargs={}, response_metadata={}, id='554b967c-bee9-4b64-a68f-bcd2c6b4b7b7'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Input text: "john Doe, john.doe@example.com, 555-1234"\n   - Task: Extract contact info\n   - Available tool: `ContactInfo` with parameters `name`, `email`, `phone` (all required)\n\n2.  **Identify Required Parameters:**\n   - `name`: "john Doe"\n   - `email`: "john.doe@example.com"\n   - `phone`: "555-1234"\n\n3.  **Validate Parameters:**\n   - All required parameters are present in the input.\n   - Format matches the expected types (strings).\n\n4.  **Construct Tool Call:**\n   - Function: `ContactInfo`\n   - Arguments: `{"name": "john Doe", "email": "john.doe@example.com", "phone": "555-1234"}`\n\n5.  **Execute Tool Call:** (Mental simulation)\n   - I will generat

In [33]:
## Dataclass
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from dataclasses import dataclass

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str
    name: str
    email: str
    phone: str


agent= create_agent(
    model, 
    response_format=ToolStrategy(ContactInfo)  # Auto-selects ProviderStrategy
    )

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact info from : john Doe, john.doe@example.com, 555-1234"
    }]
})

result["structured_response"]

ContactInfo(name='john Doe', email='john.doe@example.com', phone='555-1234')